## Notebook for visualisation and statistics of training dataset
03-07-2025, Olivier Meyers

In [ ]:
import os
# Root of the local Bedretto dataset (not distributed with this repo, see README)
DATA_DIR = os.environ.get("BEDRETTO_DATA_DIR", "data") + "/"
os.chdir(DATA_DIR)  # notebook uses paths relative to the dataset root

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import h5py
from scipy.signal import stft,istft
import obspy
from obspy import Trace, Stream
from obspy.core import UTCDateTime
import matplotlib.dates as mdates
import os
import glob
from scipy.signal.windows import tukey



In [ ]:
# Load metadata from CSV file
metadata = pd.read_csv(r"metadata.csv")
metadata.head()
#catalog = pd.read_csv(r"catalogtrain.csv"



In [ ]:
metadata.info()

In [ ]:
num_traces = len(metadata)

# Histogram of magnitudes
plt.figure(figsize=(8, 4))
plt.hist(metadata['source_magnitude'], bins=30, edgecolor='black', alpha=0.7)
plt.title(f'Magnitude Distribution (n={num_traces}, training dataset)')
plt.xlabel('Magnitude (MwA)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.show()

# Pie chart for source type
plt.figure(figsize=(6, 4))
metadata['source_type'].value_counts().plot.pie(autopct='%1.1f%%', startangle=90)
plt.title(f'Source Type Distribution (n={num_traces}, training dataset)')
plt.ylabel('')
plt.show()

# Histogram for SNR 
plt.figure(figsize=(6, 4))
plt.hist(metadata['trace_snr'], bins=40, edgecolor='black', alpha=0.7)
plt.yscale('log')
plt.title(f'Trace SNR Distribution (n={num_traces}, training dataset)')
plt.xlabel('SNR')
plt.ylabel('Frequency (log scale)')
plt.grid(True, alpha=0.3, which='both')
plt.show()

# Histogram for source depth
plt.figure(figsize=(6, 4))
plt.hist(metadata['source_depth_m'], bins=30, edgecolor='black', alpha=0.7)
plt.title(f'Source Depth Distribution (n={num_traces}, training dataset)')
plt.xlabel('Depth (m)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.show()

# Histogram for source size
plt.figure(figsize=(6, 4))
plt.hist(metadata['source_size'], bins=30, edgecolor='black', alpha=0.7)
plt.title(f'Source Size Distribution (n={num_traces}, training dataset)')
plt.xlabel('Size (dm)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.show()


# Pie chart for trace channel
plt.figure(figsize=(6, 4))
metadata['trace_channel'].value_counts().plot.pie(autopct='%1.1f%%', startangle=90)
plt.title(f'Trace Channel Distribution (n={num_traces}, training dataset)')
plt.ylabel('')
plt.show()

# Histogram for path_hyp_distance_m
plt.figure(figsize=(6, 4))
plt.hist(metadata['path_hyp_distance_m'], bins=30, edgecolor='black', alpha=0.7)
plt.title(f'Hypocentral Distance Distribution (n={num_traces}, training dataset)')
plt.xlabel('Distance (m)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.show()

# Histogram for trace_p_uncertainty_s
plt.figure(figsize=(6, 4))
plt.hist(metadata['trace_p_uncertainty_s'].dropna(), bins=30, edgecolor='black', alpha=0.7)
plt.title(f'P Arrival Uncertainty Distribution (n={num_traces}, training dataset)')
plt.xlabel('Uncertainty (s)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.show()

# Pie chart for trace_polarity (include NaN)
plt.figure(figsize=(6, 4))
polarity_with_nan = metadata['trace_polarity'].fillna('NaN/Missing') # Create a copy of the column and fill NaN with a label
polarity_with_nan.value_counts().plot.pie(autopct='%1.1f%%', startangle=90)
plt.title(f'Trace Polarity Distribution (n={num_traces}, training dataset)')
plt.ylabel('')
plt.show()

# Print polarity counts
print("Trace Polarity Distribution:")
print(polarity_with_nan.value_counts())
print(f"\nTotal entries: {len(polarity_with_nan)}")

# Scatter plot for hypocentral distance vs S arrival sample
plt.figure(figsize=(8, 6))
plt.scatter(metadata['path_hyp_distance_m'], metadata['trace_s_arrival_sample'], alpha=0.5)
plt.xlabel('Hypocentral Distance (m)')
plt.ylabel('S Arrival Sample')
plt.title(f'Correlation between S Arrival Sample and Hypocentral Distance (n={num_traces}, training dataset)')
plt.grid(True, alpha=0.3)
plt.show()

# Histogram for S wave arrival sample
plt.figure(figsize=(6, 4))
plt.hist(metadata['trace_s_arrival_sample'], bins=100, edgecolor='black', alpha=0.7)
plt.title(f'S Wave Arrival Distribution (n={num_traces}, training dataset)')
plt.xlabel('Sample number')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)
plt.show()

# Pie chart for station code
plt.figure(figsize=(6, 4))
metadata['station_code'].value_counts().plot.pie(autopct='%1.1f%%', startangle=90)
plt.title(f'Station Code Distribution (n={num_traces}, training dataset)')
plt.ylabel('')
plt.show()

# Pie chart for s wave arrival theoretical
plt.figure(figsize=(6, 4))
metadata['trace_s_arrival_theoretical'].value_counts().plot.pie(autopct='%1.1f%%', startangle=90)
plt.title(f'S Wave Arrival Theoretical Distribution (n={num_traces}, training dataset)')
plt.ylabel('')
plt.show()

# Print station counts
stations = metadata['station_code']
print("Station counts:")
print(stations.value_counts())
print(f"\nTotal entries: {len(stations)}")

# amount of unique events
source_id= metadata['source_id']
source_id.describe()




In [ ]:
# Count the number of traces with SNR above 200
num_traces_above_200 = (metadata['trace_snr'] > 3).sum()
print(f"Number of traces with SNR > 200: {num_traces_above_200}")

In [ ]:
source_magnitude_type = metadata['trace_s_arrival_theoretical']
source_magnitude_type.value_counts()

In [ ]:
s_arrival_sample= metadata['trace_s_arrival_sample']
s_arrival_sample.info()


In [ ]:
# trace start times
metadata['trace_start_datetime'] = pd.to_datetime(metadata['trace_start_time']) # Convert trace_start_time to datetime

# timeline plot
plt.figure(figsize=(15, 6))
plt.scatter(metadata['trace_start_datetime'], 
           range(len(metadata)), 
           s=1, alpha=0.7, c='blue')

plt.xlabel('Date and Time')
plt.ylabel('Trace Index')
plt.title(f'Timeline of All Trace Start Times (n={num_traces}, training dataset)')
plt.grid(True, alpha=0.3)
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d')) #formatting x-axis dates nicely
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print(f"Time range: {metadata['trace_start_datetime'].min()} to {metadata['trace_start_datetime'].max()}")


In [ ]:
#2D source locations
plt.figure(figsize=(8, 6))
plt.scatter(metadata['source_longitude'], metadata['source_latitude'], s=5, alpha=0.5)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title(f'2D Source Locations (n={num_traces}, training dataset)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#3D source locations
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')

# Size points by magnitude, color by depth
# Exponential scaling for marker sizes based on magnitude
sizes = np.exp(metadata['source_magnitude'] - metadata['source_magnitude'].min()) * 10

scatter = ax.scatter(metadata['source_longitude'], 
                    metadata['source_latitude'], 
                    metadata['source_depth_m']/1000,  # Convert to km 
                    c=metadata['source_depth_m'], 
                    cmap='viridis_r',  # Shallow=yellow, deep=purple
                    s=sizes, 
                    alpha=0.7,
                    edgecolors='black', 
                    linewidth=0.5)

# Labels and title
ax.set_xlabel('Longitude (°E)', fontsize=12)
ax.set_ylabel('Latitude (°N)', fontsize=12)
ax.set_zlabel('Depth (km below surface)', fontsize=12)
ax.set_zlim(metadata['source_depth_m'].min()/1000, metadata['source_depth_m'].max()/1000)
# Colorbar
cbar = plt.colorbar(scatter, ax=ax, shrink=0.8, aspect=20)
cbar.set_label('Depth (m)', rotation=270, labelpad=20)

# viewing angle
ax.view_init(elev=25, azim=-60)

# depth reference plane at surface
#xx, yy = np.meshgrid(
#    np.linspace(metadata['source_longitude'].min(), metadata['source_longitude'].max(), 10),
#    np.linspace(metadata['source_latitude'].min(), metadata['source_latitude'].max(), 10)
#)
#zz = np.zeros_like(xx)
#ax.plot_surface(xx, yy, zz, alpha=0.1, color='gray')
plt.title(f'3D Source Locations (n={num_traces}, training dataset)', fontsize=14)
plt.show()

Waveform Data

In [ ]:
# Inspect the structure of the HDF5 file
with h5py.File('waveforms.hdf5', 'r') as f:
    def print_structure(name, obj):
        print(name)
    f.visititems(print_structure)


In [ ]:
with h5py.File('waveforms.hdf5', 'r') as f:
    print("data_format/component_order:", f['data_format/component_order'][()])
    print("data_format/dimension_order:", f['data_format/dimension_order'][()])
    print("data_format/instrument_response:", f['data_format/instrument_response'][()])
    print("data_format/measurement:", f['data_format/measurement'][()])
    print("data_format/unit:", f['data_format/unit'][()])

In [ ]:
bucketnumber_to_plot = 0 # Change this to select a different bucket, between 0 and 16
traceindex_to_plot = 0 # Change this to select a different trace index (0 to 1999, bucket 16 only has 853 traces)

# Load data from HDF5 (convert to ObsPy format)
with h5py.File('waveforms.hdf5', 'r') as f:
    # Load waveform data
    bucket0_data = f[f'data/bucket{bucketnumber_to_plot}'][...] 
    sampling_rate = metadata['trace_sampling_rate_hz'].iloc[0]  
    
    print(f"Data shape: {bucket0_data.shape}")
    print(f"Sampling rate: {sampling_rate} Hz")
    print(f"Number of traces in bucket: {bucket0_data.shape[0]}")
    print(f"Number of components: {bucket0_data.shape[1]}")
    print(f"Number of time points: {bucket0_data.shape[2]}")

    # Extract single trace
    trace_data = bucket0_data[traceindex_to_plot, 0, :]  # (trace_index, component, time_points)
    
    print(f"Selected trace shape: {trace_data.shape}")
    
    # Create ObsPy Trace
    #tr = Trace(data=trace_data)
    #tr.stats.sampling_rate = sampling_rate
    #tr.stats.channel = "BHZ"  # Vertical component
    #tr.stats.starttime = UTCDateTime(0) #to be updated with metadata csv
    
    #st = Stream([tr])
    #print(st)

In [ ]:
# plot trace with P and S arrivals
def plot_trace_with_arrivals(sampling_rate, traceindex_to_plot, bucketnumber_to_plot, metadata):
    """Plot seismic trace with P and S arrival markers"""
    #trace_data = bucket0_data[traceindex_to_plot, 0, :]
    # Get the corresponding metadata row
    trace_name = f"bucket{bucketnumber_to_plot}${traceindex_to_plot},:1,:30001"
    trace_metadata = metadata.loc[metadata['trace_name'] == trace_name].iloc[0]
    
    # Calculate time axis
    time_axis = np.arange(len(trace_data)) / sampling_rate
    
    # Create plot
    fig, ax = plt.subplots(figsize=(5, 2))
    ax.plot(time_axis, trace_data, 'k-', linewidth=0.5, label='Seismic trace')
    
    # Add P-arrival marker 
    p_arrival_time = trace_metadata['trace_p_arrival_sample'] / sampling_rate
    ax.axvline(x=p_arrival_time, color='red', linestyle='--', linewidth=2, label='P-arrival')
    ax.text(p_arrival_time - 0.001, np.max(trace_data)*0.9, 'P', 
           ha='center', va='bottom', fontsize=12, color='red', fontweight='bold')
    
    # Add S-arrival marker
    s_arrival_time = trace_metadata['trace_s_arrival_sample'] / sampling_rate
    ax.axvline(x=s_arrival_time, color='blue', linestyle='--', linewidth=2, label='S-arrival')
    ax.text(s_arrival_time - 0.001, np.max(trace_data)*0.9, 'S', 
           ha='center', va='bottom', fontsize=12, color='blue', fontweight='bold')
     
    # Add other markers
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('Amplitude')
    ax.set_title(f'Bucket Index: {bucketnumber_to_plot}, Trace Index: {traceindex_to_plot}, Mag: {trace_metadata["source_magnitude"]:.1f}, Pol: {trace_metadata["trace_polarity"]}, SNR: {trace_metadata["trace_snr"]:.1f}, Start time: {trace_metadata["trace_start_time"]}')
    #ax.set_xlim(0.04975,0.0504)  # Set x-axis limits based on trace length
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    plt.show()


In [ ]:
import random

num_traces_to_plot = 30
for _ in range(num_traces_to_plot):
    bucket = random.randint(0, 2)
    # Find number of traces in this bucket
    num_traces = metadata[metadata['trace_name'].str.startswith(f'bucket{bucket}$')].shape[0]
    trace_idx = random.randint(0, num_traces - 1)
    # Load bucket data if not already loaded
    with h5py.File('waveforms.hdf5', 'r') as f:
        bucket_data = f[f'data/bucket{bucket}'][...]
        trace_data = bucket_data[trace_idx, 0, :]
    # Get sampling rate from metadata
    sampling_rate = metadata['trace_sampling_rate_hz'].iloc[0]
    plot_trace_with_arrivals(sampling_rate, trace_idx, bucket, metadata)

In [ ]:
#empirically trying to determine SNR formula
noise = trace_data[:10000] #noise is the first 10000 samples
event = trace_data[10000:20000] #event
SNR1 = (np.max(trace_data))/ (np.sqrt(np.mean(noise**2)))
SNR2 = (np.sqrt(np.mean(event**2)))/(np.sqrt(np.mean(noise**2))) # most probably
print(f"SNR1: {SNR1}, SNR2: {SNR2}")

In [ ]:
def seismic_stft(trace_data, sampling_rate, nperseg=250, nfft=500, boundary='zeros'):
    """
    Compute the Short-Time Fourier Transform (STFT) of a seismic trace after normalization.
    
    Parameters:
    - trace_data: 1D numpy array of seismic trace data
    - sampling_rate: Sampling rate in Hz
    - nperseg: Length of each segment for STFT
    - nfft: Number of FFT points
    - boundary: Boundary extension mode
    
    Returns:
    - f: Frequencies
    - t: Time bins
    - Zxx: STFT result 
    - magnitude: Magnitude spectrogram (dB)
    """

    # Get the corresponding metadata row
    trace_name = f"bucket{bucketnumber_to_plot}${traceindex_to_plot},:1,:20001"
    trace_metadata = metadata.loc[metadata['trace_name'] == trace_name].iloc[0]

    # Normalize: remove mean and divide by std
    trace_norm = (trace_data - np.mean(trace_data)) / np.std(trace_data)
    
    f, t, Zxx = stft(trace_norm, fs=sampling_rate, nperseg=nperseg, nfft=nfft, boundary=boundary)
    
    magnitude = np.abs(Zxx)  # Magnitude
    
    fig, ax = plt.subplots(1, figsize=(12, 3))

    # STFT Spectrogram magnitude
    log_mag = 20 * np.log10(magnitude / np.max(magnitude) + 1e-12)
    vmin = -60
    vmax = 0
    im = ax.pcolormesh(t, f, log_mag, vmin=vmin, vmax=vmax, cmap='viridis', shading='auto')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(f'Bucket Index: {bucketnumber_to_plot}, Trace Index: {traceindex_to_plot}, Mag: {trace_metadata["source_magnitude"]:.1f}, Pol: {trace_metadata["trace_polarity"]}, SNR: {trace_metadata["trace_snr"]:.1f}, Start time: {trace_metadata["trace_start_time"]}')
    ax.set_ylim(0, min(50000, f.max()))
    ax.set_xlim(0, 0.1)
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Normalized Amplitude (dB)')


    plt.tight_layout()
    plt.show()
    
    return f, t, Zxx, magnitude

# Apply to the selected trace
f, t, Zxx, magnitude = seismic_stft(trace_data, sampling_rate)
print(np.shape(Zxx))

In [ ]:
def rupture_area_mw(mw):
    """
    Wells & Coppersmith (1994) empirical relationship
    
    Parameters: Mw; moment magnitude
    Returns: area in km²
    """
    logA = -3.49 + 0.91 * mw
    area_km2 = 10 ** logA
    return area_km2 * 100000000 #km2 to dm2 conversion

print(rupture_area_mw(-2.940054701))  # Output: ~1260 km²

In [ ]:
def source_radius_from_mw(mw, stress_drop_pa = 5e6):
    """
    Brune's circular source model.
    
    Params
        mw (float): Moment magnitude
        stress_drop_pa, default 5 MPa
    
    Returns:
        Source radius in meters
    """
    M0 = 10 ** (1.5 * mw + 9.1)  # seismic moment in Nm
    r = ((7 / 16) * (M0 / stress_drop_pa)) ** (1 / 3)
    return r

In [ ]:
#calc source radius for first 10 unique source magnitudes
# first 10 unique source magnitudes
unique_mags = metadata['source_magnitude'].unique()[:10]

results = []
for mw in unique_mags:
    # find source_size for this magnitude
    source_size = metadata.loc[metadata['source_magnitude'] == mw, 'source_size'].iloc[0]
    radius = source_radius_from_mw(mw)
    results.append((mw, radius, source_size))

# Print results
print("source_magnitude | source_radius_from_mw (m) | source_size (dm?)")
for mw, radius, source_size in results:
    print(f"{mw:.4f}          | {radius:.2f}                  | {source_size}")

In [ ]:
def snr1 (trace_data):
    noise = trace_data[:10000] #noise is the first 10000 samples
    event = trace_data[10000:trace_s_arrival_sample] #event
    return (np.max(event)) / (np.sqrt(np.mean(noise**2)))   

def snr2 (trace_data):
    noise = trace_data[:10000] #noise is the first 10000 samples
    event = trace_data[10000:trace_s_arrival_sample] #event
    return (np.sqrt(np.mean(event**2))) / (np.sqrt(np.mean(noise**2)))

def snr3 (trace_data):
    noise = trace_data[:10000] #noise is the first 10000 samples
    event = trace_data[10000:trace_s_arrival_sample] #event
    return 10 * np.log10(np.max(event) / np.max(noise))  # SNR in dB

def snr4(trace_data):
    noise = trace_data[:9999]
    event = trace_data[10000:trace_s_arrival_sample]
    return np.std(event) / np.std(noise)

def snr5(trace_data):
    noise = trace_data[:9999]
    event = trace_data[10000:trace_s_arrival_sample]
    return np.max(event) / np.std(noise)



In [ ]:
# SNR values for first 10 traces in bucket 0

snr_table = []

with h5py.File('waveformstrain.hdf5', 'r') as f:
    bucket0_data = f['data/bucket0'][...]
    for i in range(10):
        trace_data_i = bucket0_data[i, 0, :]
        snr_metadata = metadata.loc[metadata['trace_name'] == f'bucket0${i},:1,:20001', 'trace_snr'].values[0]
        trace_s_arrival_sample = metadata.loc[metadata['trace_name'] == f'bucket0${i},:1,:20001', 'trace_s_arrival_sample'].values[0]
        snr1_val = snr1(trace_data_i)
        snr2_val = snr2(trace_data_i)
        snr3_val = snr3(trace_data_i)
        snr4_val = snr4(trace_data_i)
        snr5_val = snr5(trace_data_i)
        snr_table.append([i, snr_metadata, snr1_val, snr2_val, snr3_val, snr4_val, snr5_val])

#print(snr_table)
# display
snr_df = pd.DataFrame(snr_table, columns=['Trace Index', 'trace_snr (metadata)', 'snr1 (max(event)/RMS(noise))', 'snr2 (RMS)', 'snr3 (dB)','snr4 (std)', 'snr5 (max(event)/std(noise))'])
print(snr_df)

In [ ]:
bucketnumber_to_plot = 0 # Change this to select a different bucket, between 0 and 16
traceindex_to_plot = 0 # Change this to select a different trace index (0 to 1999, bucket 16 only has 853 traces)

In [ ]:
def plot_log_amplitude_vs_log_frequency(trace_data, sampling_rate, trace_index, bucket_number, metadata):
    """
    log-log plot of amplitude vs frequency

    Param:
    - trace_data: 1D numpy array of seismic trace data
    - sampling_rate: Sampling rate in Hz
    - trace_index: Index of trace within bucket
    - bucket_number: Bucket number
    - metadata: Metadata DataFrame
    """
    
    # Get metadata 
    trace_name = f"bucket{bucket_number}${trace_index},:1,:20001"
    trace_metadata = metadata.loc[metadata['trace_name'] == trace_name].iloc[0]
    
    # Normalize
    trace_norm = (trace_data - np.mean(trace_data)) / np.std(trace_data)
    
    # window function (reduce spectral leakage) and compute FFT
    windowed_data = trace_norm * np.hanning(len(trace_norm))

    # Compute FFT
    fft_result = np.fft.fft(windowed_data)
    frequencies = np.fft.fftfreq(len(windowed_data), 1/sampling_rate)
    
    # positive frequencies and amps
    positive_freq_mask = frequencies > 0
    freq_positive = frequencies[positive_freq_mask]
    amplitude_positive = np.abs(fft_result[positive_freq_mask])
    
    # Remove zero amplitudes (log(o) is undefined)
    nonzero_mask = amplitude_positive > 0
    freq_nonzero = freq_positive[nonzero_mask]
    amp_nonzero = amplitude_positive[nonzero_mask]
    
    # Create log-log plot
    plt.figure(figsize=(10, 6))
    plt.loglog(freq_nonzero, amp_nonzero, 'b-', linewidth=0.8, alpha=0.7)
    
    # Add Nyquist frequency line
    nyquist_freq = sampling_rate / 2
    plt.axvline(x=nyquist_freq, color='red', linestyle='--', linewidth=2, 
                label=f'Nyquist frequency ({nyquist_freq:.0f} Hz)')
    
    plt.xlabel('Frequency (Hz)')
    plt.ylabel('Amplitude')
    plt.title(f'Log Amplitude/Log Frequency\nBucket {bucket_number}, Trace {trace_index}, '
              f'Mag: {trace_metadata["source_magnitude"]:.1f}, SNR: {trace_metadata["trace_snr"]:.1f}')
    plt.grid(True, alpha=0.3, which='both')
    plt.legend()
    
    # frequency where amplitude drops to 10% of maximum
    max_amp = np.max(amp_nonzero)
    #threshold_10pct = 0.1 * max_amp
    #freq_10pct_idx = np.where(amp_nonzero > threshold_10pct)[0]
    #if len(freq_10pct_idx) > 0:
    #    freq_10pct = freq_nonzero[freq_10pct_idx[-1]]
    #    plt.axvline(x=freq_10pct, color='green', linestyle='-.', linewidth=2,
    #               label=f'10% amplitude cutoff ({freq_10pct:.0f} Hz)')
    
    plt.tight_layout()
    plt.show()
    
    # Print analysis summary
    print(f"\nFrequency Analysis Summary:")
    print(f"sampling rate: {sampling_rate} Hz")
    print(f"Nyquist frequency: {nyquist_freq} Hz")
    print(f"Maximum amplitude frequency: {freq_nonzero[np.argmax(amp_nonzero)]:.1f} Hz")
    #if len(freq_10pct_idx) > 0:
    #    print(f"10% amplitude cutoff: {freq_10pct:.1f} Hz")
    #    print(f"Suggested max sampling rate: {2 * freq_10pct:.1f} Hz")
    
    return freq_nonzero, amp_nonzero

# Apply to trace currently selected
freq_analysis, amp_analysis = plot_log_amplitude_vs_log_frequency(
    trace_data, sampling_rate, traceindex_to_plot, bucketnumber_to_plot, metadata)

In [ ]:
def analyze_frequency_content_multiple_traces(metadata=metadata, filename='waveformstrain.hdf5', SNR_highpass = True, SNR_highpass_freq= 1000):
    """
    Analyze frequency content across all buckets 
    params:
    - metadata: Metadata DataFrame
    """
    
    with h5py.File(filename, 'r') as f:
        sampling_rate = metadata['trace_sampling_rate_hz'].iloc[0]
        
        # Collect amplitude data from all buckets
        all_amplitudes_noise = []
        frequencies_noise = None
        all_amplitudes_event = []
        frequencies_event = None
        
        # Process all 17 buckets 
        for bucket_num in range(17):
            bucket_data = f[f'data/bucket{bucket_num}'][...]
            num_traces = bucket_data.shape[0]    
            for i in range(num_traces):
                trace_data = bucket_data[i, 0, :]
                if SNR_highpass == True:
                    # Try both possible trace name formats
                    location_20001 = f'bucket{bucket_num}${i},:1,:20001'
                    location_20000 = f'bucket{bucket_num}${i},:1,:20000'
                    
                    # Check which one exists in metadata
                    mask_20001 = metadata['trace_name'] == location_20001
                    mask_20000 = metadata['trace_name'] == location_20000
                    
                    if mask_20001.any():
                        SNR = metadata.loc[mask_20001, 'trace_snr'].values[0]
                    elif mask_20000.any():
                        SNR = metadata.loc[mask_20000, 'trace_snr'].values[0]
                    else:
                        print(f"Warning: Could not find metadata for bucket{bucket_num}${i}")
                        continue
                        
                    if SNR < SNR_highpass_freq:
                        continue
                trace_data_noise = trace_data[:10000]
                trace_data_event = trace_data[10000:20000]
                
                trace_norm_event = trace_data_event
                trace_norm_noise = trace_data_noise
                #trace_norm = (trace_data - np.mean(trace_data)) / np.std(trace_data)
                
                # FFT analysis with windowing
                windowed_data_event = trace_norm_event * np.hanning(len(trace_norm_event))
                windowed_data_noise = trace_norm_noise * np.hanning(len(trace_norm_noise))

                fft_result_event = np.fft.fft(windowed_data_event)
                fft_result_noise = np.fft.fft(windowed_data_noise)
                freq_temp_event = np.fft.fftfreq(len(windowed_data_event), 1/sampling_rate)
                freq_temp_noise = np.fft.fftfreq(len(windowed_data_noise), 1/sampling_rate)
                
                # Get positive freq
                positive_freq_mask = freq_temp_event > 0
                freq_positive_event = freq_temp_event[positive_freq_mask]
                amplitude_positive_event = np.abs(fft_result_event[positive_freq_mask])

                positive_freq_mask = freq_temp_noise > 0
                freq_positive_noise = freq_temp_noise[positive_freq_mask]
                amplitude_positive_noise = np.abs(fft_result_noise[positive_freq_mask])

                # Store frequencies reference (same for all traces)
                if frequencies_event is None:
                    frequencies_event = freq_positive_event
                all_amplitudes_event.append(amplitude_positive_event)
                
                if frequencies_noise is None:
                    frequencies_noise = freq_positive_noise
                all_amplitudes_noise.append(amplitude_positive_noise)
                
            print(f"Processed bucket {bucket_num}, trace {i+1}/{num_traces} (total {len(all_amplitudes_event)} traces)")
        
        # Convert to numpy array for easier statistics
        all_amplitudes_event = np.array(all_amplitudes_event)
        all_amplitudes_noise = np.array(all_amplitudes_noise)
        
        # Calculate percentiles for interval event
        median_amp_event = np.median(all_amplitudes_event, axis=0)
        percentile_25_event = np.percentile(all_amplitudes_event, 25, axis=0)
        percentile_75_event = np.percentile(all_amplitudes_event, 75, axis=0)
        percentile_5_event = np.percentile(all_amplitudes_event, 5, axis=0)
        percentile_95_event = np.percentile(all_amplitudes_event, 95, axis=0)
        
        # Calculate percentiles for interval noise
        median_amp_noise = np.median(all_amplitudes_noise, axis=0)
        percentile_25_noise = np.percentile(all_amplitudes_noise, 25, axis=0)
        percentile_75_noise = np.percentile(all_amplitudes_noise, 75, axis=0)
        percentile_5_noise = np.percentile(all_amplitudes_noise, 5, axis=0)
        percentile_95_noise = np.percentile(all_amplitudes_noise, 95, axis=0)

    
        # Create interval plot
        fig, ax1 = plt.subplots(1, 1, figsize=(12, 8))
        
        # Plot intervals
        ax1.fill_between(frequencies_event, percentile_5_event, percentile_95_event, alpha=0.2, color='red', label='90% percentile band - event')
        ax1.fill_between(frequencies_event, percentile_25_event, percentile_75_event, alpha=0.4, color='red', label='50% percentile band - event')
        ax1.loglog(frequencies_event, median_amp_event, 'r-', linewidth=2, label='Median amplitude event')

        ax1.fill_between(frequencies_noise, percentile_5_noise, percentile_95_noise, alpha=0.2, color='blue', label='90% percentile band - noise')
        ax1.fill_between(frequencies_noise, percentile_25_noise, percentile_75_noise, alpha=0.4, color='blue', label='50% percentile band - noise')
        ax1.loglog(frequencies_noise, median_amp_noise, 'b-', linewidth=2, label='Median amplitude noise')
        
        # Add Nyquist frequency line
        nyquist_freq = sampling_rate / 2
        ax1.axvline(x=nyquist_freq, color='red', linestyle='--', linewidth=2, 
                   label=f'Nyquist frequency ({nyquist_freq:.0f} Hz)')
        
        ax1.set_xlabel('Frequency (Hz)')
        ax1.set_ylabel('Amplitude')
        ax1.set_title(f'SNR above {SNR_highpass_freq} ({all_amplitudes_event.shape[0]} traces)')
        ax1.grid(True, alpha=0.3, which='both')
        ax1.legend()
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nAnalyzed {all_amplitudes_event.shape[0]} traces from all buckets")
        print(f"Sampling rate: {sampling_rate} Hz")
        print(f"Nyquist frequency: {nyquist_freq} Hz")

# Run 
analyze_frequency_content_multiple_traces(SNR_highpass=True, SNR_highpass_freq=1000)  # Adjust SNR threshold as needed

In [ ]:
# colors defined outside of the function (so they are constant each time)
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# magnitude bins and colors
MAG_BINS = [-4.5, -4.0, -3.5, -3.0, -2.5, -2.0]
MAG_BIN_PAIRS = [(MAG_BINS[i], MAG_BINS[i+1]) for i in range(len(MAG_BINS)-1)]

# matplotlib cm syntax to create colormap for each magnitude bin
MAG_BIN_COLORS = plt.cm.tab10.resampled(len(MAG_BIN_PAIRS)).colors

def analyze_frequency_content_by_magnitude_bins(resampled_dir, metadata, 
                                                mag_bins=MAG_BINS, 
                                                mag_bin_colors=MAG_BIN_COLORS,
                                                hypo_dist_min=40, hypo_dist_max=60,
                                                window_mode='nowindow'):
    """
    params:
    - resampled_dir: contains resampled (highpass filtered) .npz files
    - metadata: metadata DataFrame
    - mag_bins: list of magnitude bin edges
    - mag_bin_colors: predefined colors for each magnitude bin
    - hypo_dist_min: min hypocentral distance (m)
    - hypo_dist_max: max hypocentral distance (m)
    """
    
    sampling_rate = 100000  # sampling rate after resampling (200kHz --> 100kHz)
    
    # magnitude bin pairs (-4.5 to -4.0, etc)
    mag_bin_pairs = [(mag_bins[i], mag_bins[i+1]) for i in range(len(mag_bins)-1)]
    
    # store data for each magnitude bin
    bin_data = {}
    
    for bin_idx, (mag_min, mag_max) in enumerate(mag_bin_pairs):
        print(f"Processing magnitude bin {mag_min} to {mag_max}...")
        
        all_amplitudes_noise = []
        all_amplitudes_event = []
        frequencies_noise = None
        frequencies_event = None
        
        # process all 17 buckets for this magnitude bin
        for bucket_num in range(17):
            npz_path = os.path.join(resampled_dir, f'bucket_{bucket_num:02d}_highpass.npz') # path of highpass data
            
            if os.path.exists(npz_path):
                data = np.load(npz_path)
                bucket_data = data['waveforms']
                num_traces = bucket_data.shape[0]
                
                for i in range(num_traces):
                    trace_data = bucket_data[i, 0, :]
                
                    # two types of trace names ...20001 and ...20000, need to check both
                    location_20001 = f'bucket{bucket_num}${i},:1,:20001' 
                    location_20000 = f'bucket{bucket_num}${i},:1,:20000'
                    
                    mask_20001 = metadata['trace_name'] == location_20001
                    mask_20000 = metadata['trace_name'] == location_20000
                    
                    if mask_20001.any():
                        magnitude = metadata.loc[mask_20001, 'source_magnitude'].values[0]
                    elif mask_20000.any():
                        magnitude = metadata.loc[mask_20000, 'source_magnitude'].values[0]
                    else:
                        continue
                    
                    # only process traces with magnitude in relevant bin
                    if not (mag_min <= magnitude < mag_max):
                        continue
                    
                    # only process traces with specified hypocentral distance range
                    hypo_dist = metadata.loc[mask_20001 | mask_20000, 'path_hyp_distance_m'].values[0]
                    if hypo_dist_min > hypo_dist or hypo_dist_max < hypo_dist:
                        continue
                    
                    #sample indices for resampled data (half of the original)
                    noise_samples = (int(0.025 * sampling_rate)) # 5000 samples at 100kHz
                    event_samples = int(0.05 * sampling_rate)  # 5000 samples at 100kHz
                    
                    trace_data_noise = trace_data[:noise_samples]
                    trace_data_event = trace_data[noise_samples:noise_samples + event_samples]
                    
                    
                    
                    # np.hanning window used to reduce spectral leakage, because the edges of the noise and event segments (0.05s) can cause artifacts
                    if window_mode == 'hanning':
                        windowed_data_event = trace_data_event * np.hanning(len(trace_data_event))
                        windowed_data_noise = trace_data_noise * np.hanning(len(trace_data_noise))
                    elif window_mode == 'tukey':
                        windowed_data_event = trace_data_event * tukey(len(trace_data_event), alpha=0.1)  # Tukey window with 10% taper
                        windowed_data_noise = trace_data_noise * tukey(len(trace_data_noise), alpha=0.1)  # Tukey window with 10% taper
                    elif window_mode == 'nowindow':
                        windowed_data_event = trace_data_event
                        windowed_data_noise = trace_data_noise
                    else:
                        raise ValueError(f"Unknown window mode: {window_mode}. Use 'hanning', 'tukey', or 'nowindow'.")
                    
                    # discrete fourier tranform 
                    fft_result_event = np.fft.fft(windowed_data_event)
                    fft_result_noise = np.fft.fft(windowed_data_noise)
                    # the dft freququencies
                    freq_temp_event = np.fft.fftfreq(len(windowed_data_event), 1/sampling_rate)
                    freq_temp_noise = np.fft.fftfreq(len(windowed_data_noise), 1/sampling_rate)
                    
                    # Get positive frequencies
                    positive_freq_mask = freq_temp_event > 0
                    freq_positive_event = freq_temp_event[positive_freq_mask]
                    amplitude_positive_event = np.abs(fft_result_event[positive_freq_mask])

                    positive_freq_mask = freq_temp_noise > 0
                    freq_positive_noise = freq_temp_noise[positive_freq_mask]
                    amplitude_positive_noise = np.abs(fft_result_noise[positive_freq_mask])

                    # Store frequencies reference
                    if frequencies_event is None:
                        frequencies_event = freq_positive_event
                    all_amplitudes_event.append(amplitude_positive_event)
                    
                    if frequencies_noise is None:
                        frequencies_noise = freq_positive_noise
                    all_amplitudes_noise.append(amplitude_positive_noise)
        
        # Store data for this magnitude bin
        if len(all_amplitudes_event) > 0:
            all_amplitudes_event = np.array(all_amplitudes_event)
            all_amplitudes_noise = np.array(all_amplitudes_noise)
            
            bin_data[bin_idx] = {
                'mag_range': f'{mag_min} to {mag_max}',
                'frequencies_event': frequencies_event,
                'frequencies_noise': frequencies_noise,
                'median_amp_event': np.median(all_amplitudes_event, axis=0),
                'percentile_25_event': np.percentile(all_amplitudes_event, 25, axis=0),
                'percentile_75_event': np.percentile(all_amplitudes_event, 75, axis=0),
                'median_amp_noise': np.median(all_amplitudes_noise, axis=0),
                'percentile_25_noise': np.percentile(all_amplitudes_noise, 25, axis=0),
                'percentile_75_noise': np.percentile(all_amplitudes_noise, 75, axis=0),
                'n_traces': len(all_amplitudes_event),
                'color': mag_bin_colors[bin_idx]  # Use predefined colors
            }
            print(f"  Found {len(all_amplitudes_event)} traces in magnitude bin {mag_min} to {mag_max}")
        else:
            print(f"  No traces found in magnitude bin {mag_min} to {mag_max}")
    
    # Create the plot
    fig, ax = plt.subplots(1, 1, figsize=(15, 10))
    
    # Plot each magnitude bin
    for bin_idx, data in bin_data.items():
        if data['n_traces'] < 10: #skip bins with 10 or less traces
            print(f"Skipping {data['mag_range']} bin (only {data['n_traces']} traces)")
            continue

        color = data['color']
        mag_range = data['mag_range']
        n_traces = data['n_traces']
        
        # plot event data
        ax.loglog(data['frequencies_event'], data['median_amp_event'], 
                 color=color, linewidth=2, linestyle='-',
                 label=f'Event median of Mag {mag_range} ({n_traces} traces)')
        
        # plot noise data with same color but dashed
        ax.loglog(data['frequencies_noise'], data['median_amp_noise'], 
                 color=color, linewidth=2, linestyle='--',
                 label=f'Noise median of Mag {mag_range} ({n_traces} traces)')
    
    # Add nyquist frequency line
    nyquist_freq = sampling_rate / 2
    ax.axvline(x=nyquist_freq, color='red', linestyle=':', linewidth=2, 
               label=f'Nyquist frequency ({nyquist_freq:.0f} Hz)')
    
    #add Butterworth 4th order highpass filter cutoff line 
    highpass_freq = 200
    ax.axvline(x=highpass_freq, color='green', linestyle=':', linewidth=2,
                label=f'Highpass cutoff ({highpass_freq:.0f} Hz)')
    
    ax.set_ylim(10, 1e7) # limit y-axis to avoid low amplitudes form resmapling at high frequencies
    #ax.set_xlim(200,60000) # limit x-axis to avoid too low frequencies
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Amplitude')

    if window_mode == 'tukey':
        window_mode_str = 'Tukey taper (10%)'
    elif window_mode == 'hanning':
        window_mode_str = 'Hanning taper'
    elif window_mode == 'nowindow':
        window_mode_str = 'No taper applied'
    ax.set_title(f'Frequency Content by Magnitude Bins\n'
             f'Processing: {window_mode_str} | Resampled: 100kHz\n'
             f'Hypocentral Distance: {hypo_dist_min}-{hypo_dist_max}m | Segments: 0-0.025s (noise), 0.05-0.1s (event)')
    #High-pass: 200Hz (2nd order Butterworth)
    ax.grid(True, alpha=0.3, which='both')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    #total_traces = sum(data['n_traces'] for data in bin_data.values())
    #print(f"\nSummary:")
    #print(f"total traces analyzed: {total_traces}")
    #print(f"sampling rate: {sampling_rate} Hz")
    #print(f"nyquist frequency: {nyquist_freq} Hz")
    #for bin_idx, data in bin_data.items():
    #    print(f"Magnitude {data['mag_range']}: {data['n_traces']} traces")

In [ ]:
analyze_frequency_content_by_magnitude_bins(resampled_dir='npz_files_100kHz_highpass_bworder2', metadata=metadata,hypo_dist_min=0, hypo_dist_max=20, window_mode= 'nowindow')

analyze_frequency_content_by_magnitude_bins(resampled_dir='npz_files_100kHz_highpass_bworder2_zerophase', metadata=metadata,hypo_dist_min=0, hypo_dist_max=20, window_mode= 'nowindow')


In [ ]:
hypo_distances= np.arange(0, 140, 20)
hypo_pairs = [(hypo_distances[i], hypo_distances[i+1]) for i in range(len(hypo_distances)-1)]
windows=['hanning', 'tukey', 'nowindow']  # Different window modes to test

#plotting loop
for window_mode in windows:
    print(f"\nAnalyzing frequency content with {window_mode} window...")
    # Loop through hypocentral distance pairs    
    for hypo_min, hypo_max in hypo_pairs:
        print(f"Analyzing frequency content for hypocentral distance {hypo_min}-{hypo_max}m")
        analyze_frequency_content_by_magnitude_bins(
            resampled_dir='npz_files_100kHz', 
            metadata=metadata,
            hypo_dist_min=hypo_min, 
            hypo_dist_max=hypo_max,
            window_mode=window_mode
        )

In [ ]:
hypo_distances= np.arange(0, 120, 20)
hypo_distance

analyze_frequency_content_by_magnitude_bins('npz_files_100kHz_highpass_bworder2', metadata,
                                           hypo_dist_min=0, hypo_dist_max=20, window_mode='tukey')


In [ ]:
hypocentral_distance = metadata['path_hyp_distance_m'] 
magnitude = metadata['source_magnitude']

In [ ]:
from scipy.signal.windows import tukey



In [ ]:
time_series = np.ones(1000)
time_series_untouched=time_series.copy()
time_series_tukey = time_series * tukey(len(time_series), alpha=0.1)  # Apply Tukey window with 10% tapering
time_series_hanning = time_series * np.hanning(len(time_series))  # Apply Hanning window

plt.figure(figsize=(12, 6))
plt.plot(time_series_untouched, label='Original signal (amplitude=1)', color='blue', alpha=0.5)
plt.plot(time_series_tukey, label='Tukey Window (5% taper on both sides)', color='orange', alpha=0.7)
plt.plot(time_series_hanning, label='Hanning Window (weighted cosine)', color='green', alpha=0.7)
plt.legend()
plt.title('Effect of different window functions')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.show()